# Ecobici Dataset Data Augmentation

In [1]:
import pandas as pd
import numpy as np
import urllib.request

## Connection OSRM (Open Source Routing Machine)

The values in we are going to compute represent the physical length (in meters) of the fastest cycling route between each pair of stations along OpenStreetMap's road and bike-lane network.

Notice it is Fastest-route distance (not shortest-path or straight-line): OSRM optimizes routes to minimize travel time (duration) according to its bicycle profile (not specific distance in meters). The distance returned is the total length of that fastest bikeable path. For example, if a slightly longer detour uses a smooth dedicated bike lane (like ciclovía) instead of a slow cobblestone street or busy avenue, OSRM chooses the faster route and reports its longer meter distance.

Ecobici stations sit on sidewalks, plazas, or medians rather than directly on road centerlines. Before calculating any routes, OSRM snaps each station's (lon, lat) coordinate to the nearest routable street or path edge in the map graph and measures between those snapped points.

Because OSRM enforces one-way streets, contraflow bike lanes, and turn restrictions, the distance from Station A to Station B will often differ from Station B to Station A (meaning the matrix is not symmetric across its diagonal).

To calculate hundreds of thousands of routes efficiently, the road network is first transformed into a directed mathematical graph of intersections and street segments weighted by expected travel time. Searching every individual street block across the entire city for every station pair would be computationally prohibitive, so the engine organizes the network using Multi-Level Dijkstra. This approach partitions the geography into nested hierarchy levels of cells and precalculates the fastest shortcut paths connecting the boundary entry and exit points of every cell.

When executing the many-to-many matrix calculation, the routing algorithm performs a bidirectional search across these precomputed cell shortcuts. By hopping across high-level neighborhood shortcuts rather than exploring every local intersection between distant areas, the engine rapidly identifies the minimum-duration path for every origin-destination pair and sums the physical meter lengths of the underlying street segments along each winning route.

In [2]:
url = "http://download.geofabrik.de/north-america/mexico-latest.osm.pbf"
print("Downloading Mexico map... (this is a large file, please wait a few minutes)")
urllib.request.urlretrieve(url, "mexico-latest.osm.pbf")
print("Download complete!")

Download complete!


In [3]:
!docker run -t -v "%cd%":/data osrm/osrm-backend osrm-extract -p /opt/bicycle.lua /data/mexico-latest.osm.pbf
!docker run -t -v "%cd%":/data osrm/osrm-backend osrm-partition /data/mexico-latest.osrm
!docker run -t -v "%cd%":/data osrm/osrm-backend osrm-customize /data/mexico-latest.osrm

'docker' is not recognized as an internal or external command,
operable program or batch file.
'docker' is not recognized as an internal or external command,
operable program or batch file.
'docker' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import json

# Load the local JSON file
with open("stations.json", "r", encoding="utf-8") as file:
    bike_data = json.load(file)

# Access the list of stations
stations = bike_data["data"]["stations"]

# Example: Print the name and capacity of the first station
first_station = stations[0]
print(f"Station: {first_station['name']}, Capacity: {first_station['capacity']}")

Station: CE-710 Molino del Rey - Glorieta de la Lealtad, Capacity: 39


In [5]:
import time

!docker rm -f osrm-cdmx

# Start the server specifying the MLD algorithm
!docker run -d --name osrm-cdmx -p 5000:5000 -v "%cd%":/data osrm/osrm-backend osrm-routed --algorithm mld --max-table-size 1000 /data/mexico-latest.osrm

print("Waiting 15 seconds for the map to load into RAM...")
time.sleep(15)

# Verify it is running
!docker logs osrm-cdmx

'docker' is not recognized as an internal or external command,
operable program or batch file.
'docker' is not recognized as an internal or external command,
operable program or batch file.


Waiting 15 seconds for the map to load into RAM...


'docker' is not recognized as an internal or external command,
operable program or batch file.


In [6]:
!docker ps -a --filter "name=osrm-cdmx"
!docker logs osrm-cdmx

'docker' is not recognized as an internal or external command,
operable program or batch file.
'docker' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
import json
import requests
import numpy as np
import pandas as pd

# 1. Load your local stations.json file
with open("stations.json", "r", encoding="utf-8") as file:
    bike_data = json.load(file)

stations = bike_data["data"]["stations"]
n = len(stations)

station_ids = [s["station_id"] for s in stations]
coords = [f"{s['lon']},{s['lat']}" for s in stations]
coord_string = ";".join(coords)

# Pre-allocate empty NxN matrices
distances = np.zeros((n, n))
durations = np.zeros((n, n))

# 2. Query in batches of 100 source stations at a time
batch_size = 100
for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    sources = ";".join(str(i) for i in range(start, end))
    
    url = f"http://localhost:5000/table/v1/bicycle/{coord_string}?sources={sources}&annotations=duration,distance"
    res = requests.get(url).json()
    
    if res.get("code") == "Ok":
        distances[start:end, :] = res["distances"]
        durations[start:end, :] = res["durations"]
        print(f"Processed stations {start} to {end - 1}...")
    else:
        print(f"Error on batch {start}-{end}:", res)
        break

# 3. Convert to DataFrames and save to CSV
df_distances = pd.DataFrame(distances, index=station_ids, columns=station_ids)
df_durations = pd.DataFrame(durations, index=station_ids, columns=station_ids)

df_distances.to_csv("ecobici_distance_matrix_meters.csv")
df_durations.to_csv("ecobici_duration_matrix_seconds.csv")

print(f"\nDone! Final matrix shape: {df_distances.shape}")
display(df_distances.head())

ConnectionError: HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /table/v1/bicycle/-99.192508,19.416795;-99.15865559132271,19.36726591257277;-99.160395,19.363404;-99.17759,19.44304;-99.162085,19.359583;-99.175269,19.367816;-99.174447,19.36473;-99.176135,19.37099;-99.184465,19.373447;-99.172119,19.424784;-99.130918,19.433766;-99.15662247688942,19.370565558464016;-99.158847,19.376793;-99.171639,19.372474;-99.16704,19.384849;-99.164376,19.368107;-99.17801052342838,19.375543381100602;-99.183744,19.385494;-99.164538,19.372469;-99.163778,19.375397;-99.162794,19.379435;-99.18109126394484,19.382123281154726;-99.18667,19.381123;-99.17058136315518,19.367103332434564;-99.179421,19.36952;-99.183254,19.398946;-99.1835709661791,19.396339843986805;-99.170272,19.377966;-99.17118497195588,19.363698614576656;-99.172683,19.362037;-99.168219,19.375865;-99.175985,19.364916;-99.161099,19.376959;-99.185573,19.375865;-99.1782718,19.3854574;-99.16706666355822,19.37313576931919;-

In [ ]:
df_distances.to_csv('station_distances.csv')